In [1]:
import time
import osxphotos

In [2]:
photosdb = osxphotos.PhotosDB('/Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）.photoslibrary')     # Load the Photos library
# photosdb = osxphotos.PhotosDB()

In [3]:
photos = photosdb.photos()     # Get all photos

In [ ]:
movies = photosdb.photos(images=False, movies=True)
images = photosdb.photos(images=True, movies=False)

In [6]:
len(images),len(movies)

(65367, 6240)

In [7]:
def photo_to_basic_row(photo):
    return {
        "uuid": photo.uuid,
        "filename": photo.filename,
        "original_filename": photo.original_filename,
        "path": str(photo.path) if photo.path else None,
        "isphoto": photo.isphoto,
        "ismovie": photo.ismovie,
        "ismissing": photo.ismissing,
        "date": str(photo.date) if photo.date else None,
        "date_added": str(photo.date_added) if photo.date_added else None,
        "title": photo.title,
        "description": photo.description,
        "keywords": photo.keywords,
        "albums": photo.albums,
    }

rows = [photo_to_basic_row(photo) for photo in photos]

len(rows), rows[0]

(71607,
 {'uuid': '2E74EDE0-DEF4-4606-8DBC-69CBEAD25593',
  'filename': '2E74EDE0-DEF4-4606-8DBC-69CBEAD25593.mp4',
  'original_filename': 'IMG_0403.mp4',
  'path': '/Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）.photoslibrary/originals/2/2E74EDE0-DEF4-4606-8DBC-69CBEAD25593.mp4',
  'isphoto': False,
  'ismovie': True,
  'ismissing': False,
  'date': '2018-04-21 20:25:06.509058+08:00',
  'date_added': '2018-04-21 20:25:10.251436+08:00',
  'title': None,
  'description': None,
  'keywords': [],
  'albums': ['美女下藥迷姦男的']})

In [8]:
hide_photos = [
    photo for photo in photos
    if "HIDE" in photo.keywords
]

len(hide_photos)

23745

In [9]:
for photo in hide_photos[:10]:
    print(photo.uuid)
    print(photo.filename)
    print(photo.date)
    print(photo.keywords)
    print(photo.albums)
    print(photo.path)
    print("-" * 80)

20F1DB0E-99A2-4B39-9360-45858B2E5D8C
20F1DB0E-99A2-4B39-9360-45858B2E5D8C.jpeg
2019-07-16 18:08:06.099051+08:00
['HIDE']
['FB Pretty Girls']
/Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）.photoslibrary/originals/2/20F1DB0E-99A2-4B39-9360-45858B2E5D8C.jpeg
--------------------------------------------------------------------------------
AB37B8DD-B5A1-4D07-AA78-D6A301BF5BCD
AB37B8DD-B5A1-4D07-AA78-D6A301BF5BCD.jpeg
2023-09-08 09:22:05.672199+08:00
['HIDE', 'NSFW']
['（隱藏內容）TG Girls -1']
/Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）.photoslibrary/originals/A/AB37B8DD-B5A1-4D07-AA78-D6A301BF5BCD.jpeg
--------------------------------------------------------------------------------
9821E1D1-69CA-423F-8987-12E3C2D4F709
9821E1D1-69CA-423F-8987-12E3C2D4F709.png
2019-01-12 10:43:33+08:00
['HIDE']
['Sexy Girls - 妖狐']
/Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/

In [10]:
p = next(photo for photo in photos if photo.albums)

[x for x in dir(p) if "album" in x.lower() or "folder" in x.lower()]

['_albums',
 '_get_album_uuids',
 'album_info',
 'albums',
 'burst_album_info',
 'burst_albums']

In [11]:
ai = p.album_info[0]

print(type(ai))
print(ai)
print([x for x in dir(ai) if not x.startswith("_")])

<class 'osxphotos.albuminfo.AlbumInfo'>
['asdict', 'creation_date', 'end_date', 'folder_list', 'folder_names', 'library_list_order', 'owner', 'parent', 'photo_index', 'photos', 'sort_order', 'start_date', 'title', 'uuid']


In [12]:
for ai in p.album_info:
    print("album title:", ai.title)
    print("album uuid:", ai.uuid)
    print("folder_names:", ai.folder_names)
    print("parent:", ai.parent)
    print("folder_list:", ai.folder_list)
    print("-" * 80)

album title: 美女下藥迷姦男的
album uuid: BC364E3F-AA10-4317-B79B-8B4A6081E2DE
folder_names: []
parent: None
folder_list: []
--------------------------------------------------------------------------------


In [13]:
def build_library_structure(photos):
    assets = {}
    albums = {}
    folders = {}
    asset_album_links = []

    for photo in photos:
        asset_uuid = photo.uuid

        assets[asset_uuid] = {
            "uuid": photo.uuid,
            "filename": photo.filename,
            "original_filename": photo.original_filename,
            "path": str(photo.path) if photo.path else None,
            "isphoto": photo.isphoto,
            "ismovie": photo.ismovie,
            "ismissing": photo.ismissing,
            "date": str(photo.date) if photo.date else None,
            "date_added": str(photo.date_added) if photo.date_added else None,
            "title": photo.title,
            "description": photo.description,
            "keywords": list(photo.keywords),
            "album_uuids": [],
        }

        for ai in photo.album_info:
            album_uuid = ai.uuid
            folder_path = "/".join(ai.folder_names) if ai.folder_names else ""

            if album_uuid not in albums:
                albums[album_uuid] = {
                    "uuid": album_uuid,
                    "title": ai.title,
                    "folder_names": list(ai.folder_names),
                    "folder_path": folder_path,
                    "asset_uuids": [],
                }

            albums[album_uuid]["asset_uuids"].append(asset_uuid)
            assets[asset_uuid]["album_uuids"].append(album_uuid)

            asset_album_links.append({
                "asset_uuid": asset_uuid,
                "album_uuid": album_uuid,
                "album_title": ai.title,
                "folder_path": folder_path,
            })

            if folder_path:
                if folder_path not in folders:
                    folders[folder_path] = {
                        "folder_path": folder_path,
                        "folder_names": list(ai.folder_names),
                        "album_uuids": set(),
                    }

                folders[folder_path]["album_uuids"].add(album_uuid)

    for folder in folders.values():
        folder["album_uuids"] = list(folder["album_uuids"])

    return {
        "assets": assets,
        "albums": albums,
        "folders": folders,
        "asset_album_links": asset_album_links,
    }

library_structure = build_library_structure(photos)

print("assets:", len(library_structure["assets"]))
print("albums:", len(library_structure["albums"]))
print("folders:", len(library_structure["folders"]))
print("asset_album_links:", len(library_structure["asset_album_links"]))

assets: 71607
albums: 5172
folders: 35
asset_album_links: 65937


In [14]:
for folder_path, folder in sorted(library_structure["folders"].items())[:20]:
    print("FOLDER:", folder_path)
    for album_uuid in folder["album_uuids"]:
        album = library_structure["albums"][album_uuid]
        print("  ALBUM:", album["title"], "assets:", len(album["asset_uuids"]))
    print("-" * 80)

FOLDER: 4G（5G)上網iPhone 12 Pro在溫州街家中測試速度（後面加了捷運跟瑜珈教室5G的測試資料）
  ALBUM: 遠傳 下午兩點半 assets: 4
  ALBUM: 遠傳 晚上10:30 assets: 5
  ALBUM: 中華 5G/4G（比較） 晚上八點  出了台電大樓站捷運站的路口 assets: 1
  ALBUM: 中華 下午兩點半到三點 assets: 5
  ALBUM: 中華 傍晚 5G試用卡（是月租費2399的速度，幾乎沒有參考價值。服務人員還說的一副理所當然，台灣現在年輕人的想法很詭異…這是代溝吧） assets: 2
  ALBUM: 中華 5G 傍晚 捷運 台北車站剛離站車廂內 assets: 1
  ALBUM: 中華 5G 凌晨五點 家中書房書桌前（居然收到5G訊號而且很強！速度居然破表的快！）後來發現不穩定一下有5G，一下子又只有4G！ assets: 3
  ALBUM: 中華 4G 凌晨五點 家中書房書桌前 assets: 1
  ALBUM: 中華 5G 傍晚 Yoga Edition 6F教室外面飲水機旁邊 assets: 2
  ALBUM: 中華 5G 晚上八點 捷運 台電大樓站月台 assets: 1
  ALBUM: 中華 5G 晚上八點 捷運 小南門站月台 assets: 1
  ALBUM: 中華 5G 半夜12點鐘 家中書房書桌前 有時居然收到5G訊號而且很強！速度居然破表的快！但不穩定。一下有5G，一下子又只有4G！然後如果熱點分享居然也可以超過100Mbps assets: 1
  ALBUM: 中華 5G 晚上八點  羅曼羅蘭？（辛亥路五段93號）速度超級快！ assets: 1
  ALBUM: 中華 傍晚4:30 assets: 2
  ALBUM: 中華 5G 傍晚 捷運 忠孝敦化捷運站月台上 assets: 1
  ALBUM: 中華 5G 傍晚 捷運 西門站月台 assets: 1
  ALBUM: 中華 晚上10:30 assets: 8
  ALBUM: 中華 5G 傍晚 捷運 中正紀念堂-》小南門 車廂內 assets: 1
  ALBUM: 中華 5G 傍晚 Yoga Edition 5F到6F樓梯間 assets: 2
  ALBUM: 中華 5G 傍晚 捷運

In [15]:
assets_not_in_album = [
    asset for asset in library_structure["assets"].values()
    if not asset["album_uuids"]
]

len(assets_not_in_album)

7701

In [16]:
albums_not_in_folder = [
    album for album in library_structure["albums"].values()
    if not album["folder_path"]
]

len(albums_not_in_folder), albums_not_in_folder[:5]

(4277,
 [{'uuid': 'BC364E3F-AA10-4317-B79B-8B4A6081E2DE',
   'title': '美女下藥迷姦男的',
   'folder_names': [],
   'folder_path': '',
   'asset_uuids': ['2E74EDE0-DEF4-4606-8DBC-69CBEAD25593',
    'CBE6BD5F-B988-45B9-BE24-C374BE7B0E44',
    '6ECE8F0E-0AA1-4376-9646-FA53943D8260',
    'E9603031-5636-4045-B49A-90EF1C31138E']},
  {'uuid': '5A87F1D5-80F4-4FAE-95D0-726BBF55111A',
   'title': '今連續三天，同一家外送：仁川韓娘小吃店。今天搭配主食跟小菜：韓式炸醬麵、辣炒魷魚。辣炒由於因為才350 ，而且是海鮮類其實我還蠻擔心份量不夠的。本來要點蜜汁肉可是實在神秘了暫時放棄，下次再來試試！經過我的言語挑釁兩天，他終於幫我加狠辣！吃完整個嘴唇都紅腫了！過癮！',
   'folder_names': [],
   'folder_path': '',
   'asset_uuids': ['E3FFA9C9-E227-49DB-A120-3BA87F515E8D',
    'E4DE122A-3F52-4FE7-97D3-52E3E6E43698',
    '18250013-648A-48C2-9CE8-F8C90B127D08',
    '0869EA73-A664-4F4A-89DC-22BACB661E59',
    '5DFD462E-2C57-4060-ADE6-D5453168AA5B',
    'DC79DF9A-C37F-49C5-90E6-84792FC9F5CF',
    '6FD39246-BDA8-4491-A7FD-046B9BCC3AC7',
    '752FD1CA-5987-4061-8AC7-875EDD5A44B0',
    'A10EDD69-85BA-4839-A4C2-58C7146F1EA9',
    'C717534E-E035-402B-9

In [17]:
from collections import Counter, defaultdict

folder_counter = Counter()
folder_to_albums = defaultdict(set)

for photo in photos:
    for ai in photo.album_info:
        folder_path = "/".join(ai.folder_names) if ai.folder_names else ""

        if folder_path:
            folder_counter[folder_path] += 1
            folder_to_albums[folder_path].add(ai.title)

print("folders with assets:", len(folder_counter))

for folder_path, count in folder_counter.most_common():
    print(folder_path, "assets:", count, "albums:", len(folder_to_albums[folder_path]))

folders with assets: 35
NSFW assets: 11873 albums: 51
HIDE assets: 8937 albums: 187
股票 assets: 7427 albums: 301
出國旅遊/#北海道便宜團 2024年9月17~9月21日 assets: 1589 albums: 9
開箱 assets: 506 albums: 35
出國旅遊 assets: 497 albums: 2
電器壞掉（不修）/點外送 assets: 466 albums: 38
業障江家/業障江品瑩 assets: 254 albums: 46
業障江家 assets: 246 albums: 29
我 assets: 135 albums: 37
股票/#台股 #重要記事本 assets: 113 albums: 2
NSFW/AV assets: 90 albums: 6
信用卡帳單 assets: 70 albums: 22
重要文件（隱藏） assets: 68 albums: 2
4G（5G)上網iPhone 12 Pro在溫州街家中測試速度（後面加了捷運跟瑜珈教室5G的測試資料） assets: 46 albums: 22
NSFW/SPY_PRETTY_GIRLS assets: 42 albums: 8
Yoga Practice Sequences assets: 39 albums: 3
HIDE_UGLY assets: 35 albums: 9
有趣的東西--雜七雜八 assets: 30 albums: 17
業障江家/林欒菲 assets: 27 albums: 6
我小時候的照片（翻拍） assets: 22 albums: 7
電器壞掉（不修） assets: 21 albums: 3
去澳洲參加1995年國際物理奧林匹亞競賽 assets: 20 albums: 10
新聞 assets: 18 albums: 8
能量冥想＋脈輪瑜伽 線上課程 by Corey assets: 16 albums: 6
股票/#YouTube股市名嘴 #預測 #報明牌 assets: 12 albums: 2
Insta360 assets: 8 albums: 3
藏傳佛教 assets: 7 albums: 1
送貨 as

In [18]:
for folder_path in sorted(folder_to_albums.keys()):
    print("FOLDER:", folder_path)
    for album_title in sorted(folder_to_albums[folder_path]):
        print("  ALBUM:", album_title)
    print("-" * 80)

FOLDER: 4G（5G)上網iPhone 12 Pro在溫州街家中測試速度（後面加了捷運跟瑜珈教室5G的測試資料）
  ALBUM: 中華 4G 凌晨五點 家中書房書桌前
  ALBUM: 中華 5G 傍晚 Yoga Edition 5F到6F樓梯間
  ALBUM: 中華 5G 傍晚 Yoga Edition 6F教室外面飲水機旁邊
  ALBUM: 中華 5G 傍晚 捷運 中正紀念堂-》小南門 車廂內
  ALBUM: 中華 5G 傍晚 捷運 台北車站剛離站車廂內
  ALBUM: 中華 5G 傍晚 捷運 忠孝敦化捷運站月台上
  ALBUM: 中華 5G 傍晚 捷運 西門剛離站往台北車站 車廂內
  ALBUM: 中華 5G 傍晚 捷運 西門快到站 車廂內
  ALBUM: 中華 5G 傍晚 捷運 西門站月台
  ALBUM: 中華 5G 凌晨五點 家中書房書桌前（居然收到5G訊號而且很強！速度居然破表的快！）後來發現不穩定一下有5G，一下子又只有4G！
  ALBUM: 中華 5G 半夜12點鐘 家中書房書桌前 有時居然收到5G訊號而且很強！速度居然破表的快！但不穩定。一下有5G，一下子又只有4G！然後如果熱點分享居然也可以超過100Mbps
  ALBUM: 中華 5G 晚上八點  羅曼羅蘭？（辛亥路五段93號）速度超級快！
  ALBUM: 中華 5G 晚上八點 捷運 台北車站往西門剛離站車廂內
  ALBUM: 中華 5G 晚上八點 捷運 台電大樓站月台
  ALBUM: 中華 5G 晚上八點 捷運 小南門站月台
  ALBUM: 中華 5G/4G（比較） 晚上八點  出了台電大樓站捷運站的路口
  ALBUM: 中華 下午兩點半到三點
  ALBUM: 中華 傍晚 5G試用卡（是月租費2399的速度，幾乎沒有參考價值。服務人員還說的一副理所當然，台灣現在年輕人的想法很詭異…這是代溝吧）
  ALBUM: 中華 傍晚4:30
  ALBUM: 中華 晚上10:30
  ALBUM: 遠傳 下午兩點半
  ALBUM: 遠傳 晚上10:30
--------------------------------------------------------------------------------
FOLDER: Empty Albums
  AL

In [19]:
caption_assets = [
    p for p in photos
    if p.description and str(p.description).strip()
]

print("assets with caption:", len(caption_assets))

assets with caption: 728


In [20]:
for i, p in enumerate(caption_assets, start=1):
    print(f"[{i}]")
    print("uuid:", p.uuid)
    print("filename:", p.filename)
    print("isphoto:", p.isphoto, "ismovie:", p.ismovie)
    print("date:", p.date)
    print("caption:", p.description)
    print("keywords:", p.keywords)
    print("albums:", p.albums)
    print("path:", p.path)
    print("-" * 100)

[1]
uuid: 0473514D-8A73-4811-A5E3-47810A034DCA
filename: 0473514D-8A73-4811-A5E3-47810A034DCA.jpeg
isphoto: True ismovie: False
date: 2025-01-28 20:44:42.800000+08:00
caption: 垃圾江品瑩把佛跳牆放在微波爐微波不拿走。以為只有它一個「人」！完全不會為別人著想完全沒有生活紀律！垃圾！
keywords: []
albums: ['#2025年夜飯  2025年1月28日 晚上7:53開始吃。只有我跟爸媽，江品瑩只是來夾菜！而且每次夾菜都擋在我面前，也不會說借過超沒禮貌！這種垃圾人難怪在哪裡都惹人厭！還怪別人要害他還怪上輩子，這輩子個性有問題自己不知道人家跟他說他也聽不進去！年夜飯超沒禮貌！我還沒來吃，它就自己先開動，把魚夾爛了！整個年夜飯我都在爸爸聊天！聊大姑丈和新竹的往事！最後搞到9:50，媽媽才想到他還沒倒垃圾問說幾點鐘！只剩10分鐘垃圾場就要關了，這才匆匆忙忙結束年夜飯。總共聊了大約兩個小時！全程有錄音！垃圾江品瑩應該又是去拉皮，搞得人不像人鬼不像鬼！年夜飯過程發生媽媽把鍋子燒掉的日常事件！把要煮給爸爸吃，煮之前還炫耀給我看說是爸爸最愛吃的菜心湯，煮到鍋子乾掉黑掉！我是第一個聞到燒焦味，防止這場災難擴大的！真的是莫名其妙多災多難的年夜飯！而且過程中有人不屑過來，結束的原因是要倒垃圾？！全華人找不到幾個是這樣子吃年夜飯的！']
path: /Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）.photoslibrary/originals/0/0473514D-8A73-4811-A5E3-47810A034DCA.jpeg
----------------------------------------------------------------------------------------------------
[2]
uuid: 27AA4611-8B4C-48D5-9296-DE358344520A
filena

In [22]:
favorite_assets = [p for p in photos if p.favorite]
likes_assets = [p for p in photos if p.likes]

print("favorite:", len(favorite_assets))
print("likes:", len(likes_assets))
print("both:", len([p for p in photos if p.favorite and p.likes]))
print("favorite only:", len([p for p in photos if p.favorite and not p.likes]))
print("likes only:", len([p for p in photos if p.likes and not p.favorite]))

favorite: 699
likes: 0
both: 0
favorite only: 699
likes only: 0


In [23]:
favorite_assets = [
    p for p in photos
    if p.favorite
]

print("favorite assets:", len(favorite_assets))

favorite assets: 699


In [24]:
for i, p in enumerate(favorite_assets[:50], start=1):
    print(f"[{i}]")
    print("uuid:", p.uuid)
    print("filename:", p.filename)
    print("isphoto:", p.isphoto)
    print("ismovie:", p.ismovie)
    print("date:", p.date)
    print("favorite:", p.favorite)
    print("keywords:", p.keywords)
    print("caption:", p.description)
    print("albums:", p.albums)
    print()

[1]
uuid: 20F1DB0E-99A2-4B39-9360-45858B2E5D8C
filename: 20F1DB0E-99A2-4B39-9360-45858B2E5D8C.jpeg
isphoto: True
ismovie: False
date: 2019-07-16 18:08:06.099051+08:00
favorite: True
keywords: ['HIDE']
caption: None
albums: ['FB Pretty Girls']

[2]
uuid: D3026A1E-AE37-413C-AC77-CBFEC123C033
filename: D3026A1E-AE37-413C-AC77-CBFEC123C033.jpeg
isphoto: True
ismovie: False
date: 2018-07-07 11:18:32.135350+08:00
favorite: True
keywords: ['HIDE']
caption: None
albums: ['穿丁字褲跟小可愛練瑜珈']

[3]
uuid: 2EBB0B35-D411-42E5-894A-8C9D522D5BEB
filename: 2EBB0B35-D411-42E5-894A-8C9D522D5BEB.jpeg
isphoto: True
ismovie: False
date: 2018-07-20 03:22:49.104088+08:00
favorite: True
keywords: ['HIDE']
caption: None
albums: []

[4]
uuid: FF0BA910-6FB0-48B6-A096-4FA898F3B369
filename: FF0BA910-6FB0-48B6-A096-4FA898F3B369.png
isphoto: True
ismovie: False
date: 2019-08-06 19:54:15+08:00
favorite: True
keywords: ['HIDE']
caption: None
albums: ['FB Pretty Girls']

[5]
uuid: 7573916B-1C40-4CF4-8D81-C64D1B5499D4
filena

In [ ]:
# missing_photos = []
# downloaded_photos = []
# for photo in photos:
#     if photo.ismissing:
#         missing_photos.append(photo)
#     else:
#         downloaded_photos.append(photo)

In [ ]:
# len(missing_photos)

In [ ]:
# len(downloaded_photos)

In [ ]:
# missing_photos.sort(key=lambda x: x.date_added, reverse=True)
# downloaded_photos.sort(key=lambda x: x.date_added, reverse=True)

In [ ]:
# ["{}".format(x.date_added) for x in missing_photos]

In [ ]:
# ["{}".format(x.date_added) for x in downloaded_photos]

In [ ]:
# for photo in photos:
#     if "{}".format(photo.date_added) == '2025-01-15 15:38:11.807182+08:00':
#         print("Photo found: {}".format(photo))
#         break

In [ ]:
# if len(missing_photos)>0:
#     print(missing_photos[0])

In [ ]:
# photos[0]

In [ ]:
# photos[0].path